In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from solarrpy import seasonalModel

# Computing Seasonal Mean

In [3]:
# Load coefficients
df_coefficients = pd.read_parquet('../results/A1_deltas.parquet', engine='pyarrow')

# Load the dictionary of arrays
with np.load('../results/A1_Ct_arrays.npz') as data:
    Ct_arrays = {int(year): data[year] for year in data.files}

In [4]:
df = pd.read_csv("../data/CAMS_data/CAMS_data_Bologna.csv")
#df = pd.read_csv("../data/Bologna.csv")
spec = {
    'target': 'GHI',
    'coords': {
        "lat": 44.4949,
        "lon": 11.3426,
        "alt": 71
    },
    'data': df,
    'coefficients': df_coefficients
}

In [5]:
# Data and coefficients
data_all = spec['data'].copy()
data_all['date'] = pd.to_datetime(data_all['date'])
coefficients = spec['coefficients'].copy()
coefficients.set_index('Year', inplace=True)

In [6]:
coeffs_A1 = []
Yt_arrays = {}
Ybar_arrays = {}
sigma_hat_arrays = {}

for year_i in range(2013, 2023):
    mask = data_all['date'].dt.year <= year_i
    data = data_all.loc[mask].copy()
    
    print(f"\nRunning model with data up to {year_i}-12-31 ({len(data)} rows)")

    Rt = data['GHI']
    date = data['date']
    clearsky = data['clearsky']
    H0 = data['H0']
    Ct = Ct_arrays[year_i]

    #Computing alpha and beta
    ratio = Rt / Ct
    eps = 1e-3 * min(1 - ratio)

    alpha_t = np.min(1 - ratio) - eps
    beta_t  = max(1 - ratio) - min(1 - ratio) + 2 * eps

    # Handle edge cases: ensure valid domain for logarithms
    # 1 - α - R_t/C_t must be > 0 and < β
    inner_arg = 1 - alpha_t - ratio

    # Clip to valid range
    #eps = 1e-6
    inner_arg = np.clip(inner_arg, eps, beta_t - eps)

    # Apply double logarithm transformation: Y_t = log(log(β) - log(1 - α - R_t/C_t))
    log_inner = np.log(inner_arg)
    log_beta = np.log(beta_t)
    outer_arg = log_beta - log_inner

    # Ensure outer_arg > 0
    outer_arg = np.clip(outer_arg, eps, None)

    Y_t = np.log(outer_arg)
    data['Y_t'] = Y_t
    Yt_arrays[f'{year_i}'] = Y_t

    seasonal_model = seasonalModel.SeasonalModel(orders=[1], periods=[365])
    seasonal_model.fit(data=data[['Y_t', 'n']], target_col='Y_t', time_col='n', include_intercept=True)

    # 6. Extract seasonal parameters and update the dictionary all at once
    a0, a1, a2 = seasonal_model._model.params.values[:3]
    a0_err, a1_err, a2_err = seasonal_model._std_errors.values[:3]

    seasonal_mean = seasonal_model.predict(data, time_col='n')
    data['Ybar_hat'] = seasonal_mean
    Ybar_arrays[f'{year_i}'] = seasonal_mean

    coeffs_A1.append({
        'Year': year_i, 'alpha': alpha_t, 'beta': beta_t,
        'delta0': coefficients.loc[year_i]['delta0'], 'delta1': coefficients.loc[year_i]['delta1'], 'delta2': coefficients.loc[year_i]['delta2'], 'delta3': coefficients.loc[year_i]['delta3'],
        'delta0_err': coefficients.loc[year_i]['delta0_err'], 'delta1_err': coefficients.loc[year_i]['delta1_err'], 'delta2_err': coefficients.loc[year_i]['delta2_err'], 'delta3_err': coefficients.loc[year_i]['delta3_err'],
        'a0': a0, 'a1': a1, 'a2': a2,
        'a0_err': a0_err, 'a1_err': a1_err, 'a2_err': a2_err,
    })

tableA1_df = pd.DataFrame(coeffs_A1)


Running model with data up to 2013-12-31 (3287 rows)

Running model with data up to 2014-12-31 (3652 rows)

Running model with data up to 2015-12-31 (4017 rows)

Running model with data up to 2016-12-31 (4383 rows)

Running model with data up to 2017-12-31 (4748 rows)

Running model with data up to 2018-12-31 (5113 rows)

Running model with data up to 2019-12-31 (5478 rows)

Running model with data up to 2020-12-31 (5844 rows)

Running model with data up to 2021-12-31 (6209 rows)

Running model with data up to 2022-12-31 (6574 rows)


# Variance Estimation

In [18]:
coeffs_A2 = []
for year_i in range(2013, 2023):
    mask = data_all['date'].dt.year <= year_i
    data = data_all.loc[mask].copy()
    
    print(f"\nRunning model with data up to {year_i}-12-31 ({len(data)} rows)")
    
    Y_t = Yt_arrays[f'{year_i}']
    
    # Compute the increments of Y_t and square them
    Y_t_increments = Y_t.diff()
    
    sigma_t = Y_t_increments ** 2
    data['sigma_t'] = sigma_t

    # Create a mathematically clean subset specifically for the regression
    data_clean = data.dropna(subset=['sigma_t', 'n'])

    # Regress sigma_t on n
    model_sigma = seasonalModel.SeasonalModel(orders=[1], periods=[365])
    model_sigma.fit(data=data_clean[['sigma_t', 'n']], target_col='sigma_t', time_col='n', include_intercept=True)

    # Compute the estimated variance
    sigma_hat = model_sigma.predict(data, time_col='n')

    b0, b1, b2 = model_sigma._model.params.values[:3]
    b0_err, b1_err, b2_err = model_sigma._std_errors.values[:3]

    # Store with the joining key (Year)
    coeffs_A2.append({
        'Year': year_i,
        'b0': b0,
        'b1': b1,
        'b2': b2,
        'b0_err': b0_err,
        'b1_err': b1_err,
        'b2_err': b2_err
    })

# 1. Construct the secondary DataFrame once
tableA2_df = pd.DataFrame(coeffs_A2)


Running model with data up to 2013-12-31 (3287 rows)

Running model with data up to 2014-12-31 (3652 rows)

Running model with data up to 2015-12-31 (4017 rows)

Running model with data up to 2016-12-31 (4383 rows)

Running model with data up to 2017-12-31 (4748 rows)

Running model with data up to 2018-12-31 (5113 rows)

Running model with data up to 2019-12-31 (5478 rows)

Running model with data up to 2020-12-31 (5844 rows)

Running model with data up to 2021-12-31 (6209 rows)

Running model with data up to 2022-12-31 (6574 rows)


In [19]:
# 6. Reorder the columns so 'Year' is the very first column
cols = ['Year'] + [col for col in tableA1_df.columns if col != 'Year']
tableA1_df = tableA1_df[cols]
#tableA1_df.to_csv('../results/TableA1.csv', index=False)

In [20]:
df = tableA1_df

# 1. Format the 'Train years' column
df['Train years'] = '2005-' + df['Year'].astype(str)

# 2. Add 'Obs.' (using the values from your screenshot)
df['Obs.'] = [3287, 3652, 4017, 4383, 4748, 5113, 5478, 5844, 6209, 6574]

# 3. Format the combined columns (Value + Standard Error with HTML <br>)
df['a_0_fmt'] = df.apply(lambda row: f"{row['a0']:.4f}<br>({row['a0_err']:.4f})", axis=1)
df['a_1_fmt'] = df.apply(lambda row: f"{row['a1']:.4f}<br>({row['a1_err']:.4f})", axis=1)
df['a_2_fmt'] = df.apply(lambda row: f"{row['a2']:.4f}<br>({row['a2_err']:.4f})", axis=1)

# 3. Format the combined columns (Value + Standard Error with HTML <br>)
df['delta_0_fmt'] = df.apply(lambda row: f"{row['delta0']:.4f}<br>({row['delta0_err']:.4f})", axis=1)
df['delta_1_fmt'] = df.apply(lambda row: f"{row['delta1']:.4f}<br>({row['delta1_err']:.4f})", axis=1)
df['delta_2_fmt'] = df.apply(lambda row: f"{row['delta2']:.4f}<br>({row['delta2_err']:.4f})", axis=1)
df['delta_3_fmt'] = df.apply(lambda row: f"{row['delta3']:.4f}<br>({row['delta3_err']:.4f})", axis=1)

# Format the remaining columns to standard decimal lengths
df['alpha_fmt'] = df['alpha'].apply(lambda x: f"{x:.6f}")
df['beta_fmt'] = df['beta'].apply(lambda x: f"{x:.3f}")

# 4. Select and rename columns for the final display
display_df = df[['Train years', 'Obs.', 'alpha_fmt', 'beta_fmt', 
                    'delta_0_fmt', 'delta_1_fmt', 'delta_2_fmt', 'delta_3_fmt', 
                    'a_0_fmt', 'a_1_fmt', 'a_2_fmt']]

display_df.columns = ['Train years', 'N', 'alpha', 'beta', 
                    'delta0', 'delta1', 'delta2', 'delta3', 
                    'a0', 'a1', 'a2']

# Apply CSS Styling to mimic the LaTeX academic look
styles = [
    # Top and bottom thick horizontal rules
    {'selector': 'thead th', 'props': 'border-bottom: 1px solid black; border-top: 2px solid black; text-align: center; padding: 8px;'},
    {'selector': 'tbody tr:last-child td', 'props': 'border-bottom: 2px solid black;'},
    
    # Center all text and add vertical padding
    {'selector': 'tbody td', 'props': 'text-align: center; vertical-align: middle; padding: 8px;'},
    
    # Left-align the first column (Train years)
    {'selector': 'tbody td:nth-child(1)', 'props': 'text-align: left;'},
    
    # Selective vertical lines (matching the screenshot)
    # Column 3 is alpha, Column 5 is delta_0, Column 9 is a_0
    {'selector': 'th:nth-child(3), td:nth-child(3)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(5), td:nth-child(5)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(9), td:nth-child(9)', 'props': 'border-left: 1px solid black;'}
]

# Hide the default index and render
styled_table = display_df.style.set_table_styles(styles).hide(axis='index')
display(styled_table)

Train years,N,alpha,beta,delta0,delta1,delta2,delta3,a0,a1,a2
2005-2013,3287,0.004062,0.918,-1.1711(0.3165),0.9459(0.0425),0.0111(0.0315),0.5090(0.1810),-0.0692(0.0168),-0.0737(0.0237),-0.3959(0.0237)
2005-2014,3652,0.006443,0.916,-1.0812(0.2991),0.9359(0.0401),0.0251(0.0298),0.4503(0.1711),-0.0775(0.0158),-0.0681(0.0223),-0.3913(0.0223)
2005-2015,4017,0.006947,0.915,-1.3119(0.2829),0.9672(0.0379),0.0005(0.0281),0.5852(0.1618),-0.0684(0.0150),-0.0607(0.0212),-0.3797(0.0212)
2005-2016,4383,0.005811,0.916,-1.2374(0.2727),0.9567(0.0366),0.0062(0.0271),0.5410(0.1560),-0.0713(0.0144),-0.0662(0.0204),-0.3779(0.0203)
2005-2017,4748,0.006773,0.915,-1.1559(0.2614),0.9456(0.0351),0.0190(0.0260),0.4960(0.1495),-0.0511(0.0138),-0.0637(0.0195),-0.3740(0.0194)
2005-2018,5113,0.006145,0.916,-1.1269(0.2530),0.9406(0.0339),0.0223(0.0252),0.4835(0.1447),-0.0530(0.0133),-0.0765(0.0188),-0.3812(0.0187)
2005-2019,5478,0.007438,0.915,-1.2977(0.2447),0.9632(0.0328),0.0125(0.0243),0.5876(0.1400),-0.0407(0.0128),-0.0730(0.0181),-0.3671(0.0181)
2005-2020,5844,0.008449,0.914,-1.3211(0.2362),0.9674(0.0317),0.0116(0.0235),0.6011(0.1351),-0.0290(0.0124),-0.0641(0.0175),-0.3593(0.0175)
2005-2021,6209,0.008545,0.914,-1.4942(0.2298),0.9916(0.0308),-0.0008(0.0229),0.7003(0.1315),-0.0292(0.0119),-0.0591(0.0168),-0.3571(0.0168)
2005-2022,6574,0.008311,0.914,-1.6624(0.2244),1.0146(0.0301),-0.0168(0.0223),0.7958(0.1283),-0.0221(0.0115),-0.0533(0.0163),-0.3544(0.0163)


In [21]:
# 6. Reorder the columns so 'Year' is the very first column
cols = ['Year'] + [col for col in tableA2_df.columns if col != 'Year']
tableA2_df = tableA2_df[cols]
#tableA2_df.to_csv('../results/TableA2.csv', index=False)

In [22]:
df = tableA2_df

# 1. Format the 'Train years' column
df['Train years'] = '2005-' + df['Year'].astype(str)

# 2. Add 'Obs.' (using the values from your screenshot)
df['Obs.'] = [3287, 3652, 4017, 4383, 4748, 5113, 5478, 5844, 6209, 6574]

# 3. Format the combined columns (Value + Standard Error with HTML <br>)
df['b_0_fmt'] = df.apply(lambda row: f"{row['b0']:.4f}<br>({row['b0_err']:.4f})", axis=1)
df['b_1_fmt'] = df.apply(lambda row: f"{row['b1']:.4f}<br>({row['b1_err']:.4f})", axis=1)
df['b_2_fmt'] = df.apply(lambda row: f"{row['b2']:.4f}<br>({row['b2_err']:.4f})", axis=1)

# 4. Select and rename columns for the final display
display_df = df[['Train years', 'Obs.', 'b_0_fmt', 'b_1_fmt', 'b_2_fmt']]

display_df.columns = ['Train years', 'N', 'b0', 'b1', 'b2']

# Apply CSS Styling to mimic the LaTeX academic look
styles = [
    # Top and bottom thick horizontal rules
    {'selector': 'thead th', 'props': 'border-bottom: 1px solid black; border-top: 2px solid black; text-align: center; padding: 8px;'},
    {'selector': 'tbody tr:last-child td', 'props': 'border-bottom: 2px solid black;'},
    
    # Center all text and add vertical padding
    {'selector': 'tbody td', 'props': 'text-align: center; vertical-align: middle; padding: 8px;'},
    
    # Left-align the first column (Train years)
    {'selector': 'tbody td:nth-child(1)', 'props': 'text-align: left;'},
    
    # Selective vertical lines (matching the screenshot)
    # Column 3 is alpha, Column 5 is delta_0, Column 9 is a_0
    {'selector': 'th:nth-child(3), td:nth-child(3)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(5), td:nth-child(5)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(9), td:nth-child(9)', 'props': 'border-left: 1px solid black;'}
]

# Hide the default index and render
styled_table = display_df.style.set_table_styles(styles).hide(axis='index')
display(styled_table)

Train years,N,b0,b1,b2
2005-2013,3287,1.1385(0.0770),0.3308(0.1089),0.6959(0.1089)
2005-2014,3652,1.1251(0.0658),0.2997(0.0931),0.6619(0.0931)
2005-2015,4017,1.1130(0.0601),0.2743(0.0851),0.6434(0.0851)
2005-2016,4383,1.1255(0.0572),0.2729(0.0809),0.6578(0.0809)
2005-2017,4748,1.1186(0.0530),0.2713(0.0749),0.6715(0.0749)
2005-2018,5113,1.1275(0.0512),0.2775(0.0724),0.6900(0.0723)
2005-2019,5478,1.1191(0.0473),0.2479(0.0669),0.6487(0.0669)
2005-2020,5844,1.1060(0.0441),0.2212(0.0624),0.6254(0.0624)
2005-2021,6209,1.0967(0.0418),0.2017(0.0591),0.6202(0.0590)
2005-2022,6574,1.0793(0.0398),0.1872(0.0563),0.6029(0.0562)
